# PyPSA-Earth Netzwerkanalyse: Vergleich zweier Szenarien

Diese Vorlage lädt zwei PyPSA-Earth-Netzwerke ein und bereitet eine saubere Basis für spätere Auswertungen vor.

Die Notebook-Struktur ist bewusst schlank gehalten:
1. Pakete importieren  
2. zentrale Pfade definieren  
3. zwei Netzwerke laden  
4. optionale Zusatzdateien laden  
5. kurze Plausibilitätsübersicht anzeigen  

Weitere Analyse- und Plot-Blöcke können später darunter eingefügt werden.

## 1. Pakete importieren

In [ ]:
import yaml
import pypsa
import warnings
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter
from matplotlib.patches import Patch
import geopandas as gpd
import numpy as np
import pandas as pd
from pathlib import Path
import seaborn as sns
from datetime import datetime
from cartopy import crs as ccrs
from pypsa.plot import add_legend_circles, add_legend_lines, add_legend_patches
import os
import xarray as xr
import cartopy
import glob
import logging

# ------------------------------------------------------------
# Notebook-Einstellungen
# ------------------------------------------------------------
warnings.simplefilter(action="ignore", category=FutureWarning)
warnings.simplefilter(action="ignore", category=UserWarning)

logging.getLogger("pypsa.io").setLevel(logging.ERROR)

plt.rcParams["figure.dpi"] = 120
pd.options.display.max_columns = 100
pd.options.display.max_rows = 100

## 2. Zentraler Pfadblock

Hier werden alle Pfade eingetragen, die für die Analyse benötigt werden.

Wichtig:
- `NETWORK_PATH_SCENARIO_A` und `NETWORK_PATH_SCENARIO_B` sind Pflicht.
- Alle anderen Dateien sind optional und können leer bleiben, falls sie für die spätere Analyse nicht benötigt werden.
- Die Pfade können als Windows-WSL-Pfade (`/mnt/c/...`) oder Linux-Pfade eingetragen werden.

In [ ]:
# ============================================================
# ZENTRALER PFADBLOCK
# ============================================================

# Projektordner deines PyPSA-Earth-Forks
PROJECT_ROOT = Path("/mnt/c/Users/nikla/Documents/Git/pypsa-earth-geothermal")

# Namen der Szenarien für Tabellen, Plots und spätere Vergleiche
SCENARIO_A_NAME = "EGS"
SCENARIO_B_NAME = "without EGS"

# ------------------------------------------------------------
# Pflicht: Netzwerkdateien
# ------------------------------------------------------------

NETWORK_PATH_SCENARIO_A = Path(
    "/mnt/c/Users/nikla/Documents/Git/pypsa-earth-geothermal/results/SCENARIO_A/postnetworks/elec_s_10_ec_lcopt_Co2L0.1-Ep_144h_2050_0.0514_AG_0export_393import.nc"
)

NETWORK_PATH_SCENARIO_B = Path(
    "/mnt/c/Users/nikla/Documents/Git/pypsa-earth-geothermal/results/SCENARIO_B/postnetworks/elec_s_10_ec_lcopt_Co2L0.1-Ep_144h_2050_0.0514_AG_0export_393import.nc"
)

# ------------------------------------------------------------
# Optional: Config-Dateien
# ------------------------------------------------------------

CONFIG_PATH_SCENARIO_A = Path(
    "/mnt/c/Users/nikla/Documents/Git/pypsa-earth-geothermal/results/SCENARIO_A/configs/config.yaml"
)

CONFIG_PATH_SCENARIO_B = Path(
    "/mnt/c/Users/nikla/Documents/Git/pypsa-earth-geothermal/results/SCENARIO_B/configs/config.yaml"
)

# ------------------------------------------------------------
# Optional: Geodaten
# ------------------------------------------------------------

COUNTRY_SHAPES_PATH = Path(
    "/mnt/c/Users/nikla/Documents/Git/pypsa-earth-geothermal/resources/SCENARIO_A/shapes/country_shapes.geojson"
)

GADM_SHAPES_PATH = Path(
    "/mnt/c/Users/nikla/Documents/Git/pypsa-earth-geothermal/resources/SCENARIO_A/shapes/gadm_shapes.geojson"
)

REGIONS_ONSHORE_PATH = Path(
    "/mnt/c/Users/nikla/Documents/Git/pypsa-earth-geothermal/resources/SCENARIO_A/bus_regions/regions_onshore_elec_s_10.geojson"
)

REGIONS_OFFSHORE_PATH = Path(
    "/mnt/c/Users/nikla/Documents/Git/pypsa-earth-geothermal/resources/SCENARIO_A/bus_regions/regions_offshore_elec_s_10.geojson"
)

# ------------------------------------------------------------
# Optional: Profil- und Ressourcen-Dateien
# ------------------------------------------------------------

SOLAR_PROFILE_PATH = Path(
    "/mnt/c/Users/nikla/Documents/Git/pypsa-earth-geothermal/resources/SCENARIO_A/renewable_profiles/profile_solar.nc"
)

ONWIND_PROFILE_PATH = Path(
    "/mnt/c/Users/nikla/Documents/Git/pypsa-earth-geothermal/resources/SCENARIO_A/renewable_profiles/profile_onwind.nc"
)

OFFWIND_AC_PROFILE_PATH = Path(
    "/mnt/c/Users/nikla/Documents/Git/pypsa-earth-geothermal/resources/SCENARIO_A/renewable_profiles/profile_offwind-ac.nc"
)

OFFWIND_DC_PROFILE_PATH = Path(
    "/mnt/c/Users/nikla/Documents/Git/pypsa-earth-geothermal/resources/SCENARIO_A/renewable_profiles/profile_offwind-dc.nc"
)

HYDRO_PROFILE_PATH = Path(
    "/mnt/c/Users/nikla/Documents/Git/pypsa-earth-geothermal/resources/SCENARIO_A/renewable_profiles/profile_hydro.nc"
)

# ------------------------------------------------------------
# Optional: Ausgabeordner für spätere Grafiken und Tabellen
# ------------------------------------------------------------

OUTPUT_DIR = Path("./outputs_scenario_comparison")

## 3. Hilfsfunktionen zum Laden von Dateien

Die folgenden Funktionen prüfen zuerst, ob eine Datei existiert. Dadurch bricht das Notebook nicht sofort ab, wenn optionale Dateien noch nicht eingetragen sind.

In [ ]:
def path_exists(path):
    """Returns True if path is a valid existing file or directory."""
    if path is None:
        return False
    path = Path(path)
    return str(path).strip() != "" and path.exists()


def require_file(path, label):
    """Raises a clear error if a required file does not exist."""
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(
            f"{label} wurde nicht gefunden:{path}"
            "Bitte den Pfad im zentralen Pfadblock prüfen."
        )
    return path


def load_yaml_optional(path):
    """Loads a YAML file if it exists. Otherwise returns None."""
    if not path_exists(path):
        return None

    with open(path, "r", encoding="utf-8") as f:
        return yaml.safe_load(f)


def load_geodata_optional(path):
    """Loads a GeoDataFrame if the file exists. Otherwise returns None."""
    if not path_exists(path):
        return None

    return gpd.read_file(path)


def load_xarray_optional(path):
    """Loads a NetCDF dataset if the file exists. Otherwise returns None."""
    if not path_exists(path):
        return None

    return xr.open_dataset(path)

## 4. Netzwerke laden

Nach diesem Block stehen beide Netzwerke als `n_a` und `n_b` zur Verfügung.

Zusätzlich wird ein Dictionary `networks` erzeugt. Das ist praktisch für spätere Schleifen über beide Szenarien.

In [ ]:
# Pflichtdateien prüfen
NETWORK_PATH_SCENARIO_A = require_file(NETWORK_PATH_SCENARIO_A, "Netzwerk Szenario A")
NETWORK_PATH_SCENARIO_B = require_file(NETWORK_PATH_SCENARIO_B, "Netzwerk Szenario B")

# Netzwerke laden
n_a = pypsa.Network(NETWORK_PATH_SCENARIO_A)
n_b = pypsa.Network(NETWORK_PATH_SCENARIO_B)

# Praktische Container für spätere Vergleiche
networks = {
    SCENARIO_A_NAME: n_a,
    SCENARIO_B_NAME: n_b,
}

network_paths = {
    SCENARIO_A_NAME: NETWORK_PATH_SCENARIO_A,
    SCENARIO_B_NAME: NETWORK_PATH_SCENARIO_B,
}

print("Netzwerke erfolgreich geladen:")
for scenario_name, network_path in network_paths.items():
    print(f"- {scenario_name}: {network_path}")

## 5. Optionale Zusatzdateien laden

Dieser Block lädt Configs, Geodaten und Ressourcenprofile, falls die Pfade existieren.

Wenn eine Datei nicht existiert, wird einfach `None` gespeichert. Dadurch kannst du später gezielt prüfen, ob die jeweilige Datei verfügbar ist.

In [ ]:
# ------------------------------------------------------------
# Configs
# ------------------------------------------------------------

cfg_a = load_yaml_optional(CONFIG_PATH_SCENARIO_A)
cfg_b = load_yaml_optional(CONFIG_PATH_SCENARIO_B)

configs = {
    SCENARIO_A_NAME: cfg_a,
    SCENARIO_B_NAME: cfg_b,
}

# ------------------------------------------------------------
# Geodaten
# ------------------------------------------------------------

country_shapes = load_geodata_optional(COUNTRY_SHAPES_PATH)
gadm_shapes = load_geodata_optional(GADM_SHAPES_PATH)
regions_onshore = load_geodata_optional(REGIONS_ONSHORE_PATH)
regions_offshore = load_geodata_optional(REGIONS_OFFSHORE_PATH)

geo_data = {
    "country_shapes": country_shapes,
    "gadm_shapes": gadm_shapes,
    "regions_onshore": regions_onshore,
    "regions_offshore": regions_offshore,
}

# ------------------------------------------------------------
# Ressourcenprofile
# ------------------------------------------------------------

solar_profile = load_xarray_optional(SOLAR_PROFILE_PATH)
onwind_profile = load_xarray_optional(ONWIND_PROFILE_PATH)
offwind_ac_profile = load_xarray_optional(OFFWIND_AC_PROFILE_PATH)
offwind_dc_profile = load_xarray_optional(OFFWIND_DC_PROFILE_PATH)
hydro_profile = load_xarray_optional(HYDRO_PROFILE_PATH)

profiles = {
    "solar": solar_profile,
    "onwind": onwind_profile,
    "offwind_ac": offwind_ac_profile,
    "offwind_dc": offwind_dc_profile,
    "hydro": hydro_profile,
}

# ------------------------------------------------------------
# Ausgabeordner anlegen
# ------------------------------------------------------------

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Optionale Dateien geladen, soweit vorhanden.")
print(f"Ausgabeordner: {OUTPUT_DIR.resolve()}")

## 6. Plausibilitätsübersicht

Dieser Block zeigt eine kurze Übersicht beider Netzwerke. So kannst du direkt prüfen, ob die richtigen Dateien geladen wurden.

In [ ]:
def network_overview(network, scenario_name):
    """Creates a compact overview table for a PyPSA network."""
    objective = getattr(network, "objective", np.nan)

    return {
        "Szenario": scenario_name,
        "Snapshots": len(network.snapshots),
        "Start": network.snapshots.min() if len(network.snapshots) > 0 else None,
        "Ende": network.snapshots.max() if len(network.snapshots) > 0 else None,
        "Buses": len(network.buses),
        "Generators": len(network.generators),
        "Links": len(network.links),
        "Stores": len(network.stores),
        "Storage Units": len(network.storage_units),
        "Loads": len(network.loads),
        "Lines": len(network.lines),
        "Objective": objective,
    }


overview = pd.DataFrame(
    [
        network_overview(n, scenario_name)
        for scenario_name, n in networks.items()
    ]
)

display(overview)

## 7. Verfügbare Carrier prüfen

Dieser Block zeigt, welche Carrier in beiden Netzwerken vorkommen. Das ist hilfreich, bevor später technologiespezifische Auswertungen erstellt werden.

In [ ]:
def collect_carriers(network):
    """Collects carriers from the most relevant PyPSA component tables."""
    carrier_parts = []

    for component_name in ["buses", "generators", "links", "stores", "storage_units", "loads"]:
        table = getattr(network, component_name, None)

        if table is None or table.empty or "carrier" not in table.columns:
            continue

        carriers = (
            table["carrier"]
            .dropna()
            .astype(str)
            .replace("", np.nan)
            .dropna()
            .unique()
            .tolist()
        )

        carrier_parts.extend(carriers)

    return sorted(set(carrier_parts))


carrier_overview = pd.DataFrame({
    scenario_name: pd.Series(collect_carriers(network))
    for scenario_name, network in networks.items()
})

display(carrier_overview)

## 8. Startpunkt für weitere Analyseblöcke

Ab hier kannst du später weitere Codeblöcke einfügen, zum Beispiel:
- installierte Leistung nach Technologie
- Stromerzeugung nach Carrier
- Geothermie-Erzeugung nach Strom und Wärme
- H2-Importe und inländische H2-Produktion
- Karten für regionale Kapazitäten
- Kosten- und Systemvergleich zwischen beiden Szenarien